In [ ]:
!pip install --upgrade pip setuptools wheel

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


KeyboardInterrupt: 

In [ ]:
!pip install -q datasets transformers seqeval accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import unicodedata
import numpy as np
import torch
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/indic-ner"
CKPT_DIR = f"{DRIVE_DIR}/indicbert_hi_best"

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForTokenClassification.from_pretrained(CKPT_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
id2label = model.config.id2label

processed = load_from_disk(f"{DRIVE_DIR}/naamapadam_hi_100k")
val = processed["validation"]
tokens_raw = val["tokens"]
print("Validation rows:", len(val))

Mounted at /content/drive


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Validation rows: 13460


In [ ]:
MAX_LEN = 128

def tokenize_and_align(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    )
    aligned = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labels, prev = [], None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev:
                labels.append(tags[wid])
            else:
                labels.append(-100)
            prev = wid
        aligned.append(labels)
    enc["labels"] = aligned
    return enc

val_tok = val.map(
    tokenize_and_align,
    batched=True,
    remove_columns=val.column_names,
    desc="Tokenizing",
)

from transformers import DataCollatorForTokenClassification, Trainer

trainer = Trainer(model=model, data_collator=DataCollatorForTokenClassification(tokenizer))
out = trainer.predict(val_tok)
preds = np.argmax(out.predictions, axis=-1)

y_true, y_pred = [], []
for pred, lab in zip(preds, out.label_ids):
    keep = lab != -100
    y_true.append([id2label[l] for l in lab[keep]])
    y_pred.append([id2label[p] for p in pred[keep]])

assert all(len(a) == len(b) == len(t) for a, b, t in zip(y_true, y_pred, tokens_raw))
print("Predictions aligned:", len(y_true))

Tokenizing:   0%|          | 0/13460 [00:00<?, ? examples/s]

Predictions aligned: 13460


In [ ]:
def bio_spans(tags: list[str]) -> set[tuple[int, int, str]]:
    spans, start, typ = set(), None, None
    for i, t in enumerate(tags):
        if t.startswith("B-"):
            if start is not None:
                spans.add((start, i, typ))
            start, typ = i, t[2:]
        elif t.startswith("I-"):
            if start is None or t[2:] != typ:
                if start is not None:
                    spans.add((start, i, typ))
                start, typ = i, t[2:]
        else:
            if start is not None:
                spans.add((start, i, typ))
                start, typ = None, None
    if start is not None:
        spans.add((start, len(tags), typ))
    return spans

In [ ]:
from collections import Counter, defaultdict

def overlaps(a, b) -> bool:
    return a[0] < b[1] and b[0] < a[1]

records = []

for doc_id, (true_tags, pred_tags) in enumerate(zip(y_true, y_pred)):
    gold = bio_spans(true_tags)
    pred = bio_spans(pred_tags)
    matched_gold, matched_pred = set(), set()

    for p in pred:
        exact = [g for g in gold if g == p]
        if exact:
            records.append({"doc": doc_id, "cat": "correct", "type": p[2],
                            "span": p, "gold_span": p})
            matched_gold.update(exact)
            matched_pred.add(p)
            continue
        same_extent = [g for g in gold if g[0] == p[0] and g[1] == p[1]]
        if same_extent:
            g = same_extent[0]
            records.append({"doc": doc_id, "cat": "type_error", "type": p[2],
                            "gold_type": g[2], "span": p, "gold_span": g})
            matched_gold.add(g)
            matched_pred.add(p)
            continue
        ov = [g for g in gold if overlaps(p, g)]
        if ov:
            g = max(ov, key=lambda g: min(p[1], g[1]) - max(p[0], g[0]))
            cat = "boundary_error" if g[2] == p[2] else "type_and_boundary_error"
            records.append({"doc": doc_id, "cat": cat, "type": p[2],
                            "gold_type": g[2], "span": p, "gold_span": g})
            matched_gold.add(g)
            matched_pred.add(p)
        else:
            records.append({"doc": doc_id, "cat": "spurious", "type": p[2],
                            "span": p, "gold_span": None})

    for g in gold - matched_gold:
        records.append({"doc": doc_id, "cat": "missed", "type": g[2],
                        "span": None, "gold_span": g})

cat_counts = Counter(r["cat"] for r in records)
total_pred_side = sum(v for k, v in cat_counts.items() if k != "missed")
print(f"Total predicted spans: {total_pred_side:,}")
for cat, n in cat_counts.most_common():
    print(f"{cat:26s} {n:>7,}")

Total predicted spans: 31,841
correct                     24,450
boundary_error               3,310
spurious                     2,844
missed                       2,284
type_and_boundary_error        632
type_error                     605


In [ ]:
per_type = defaultdict(Counter)
for r in records:
    key = r["type"] if r["cat"] != "missed" else r["type"]
    per_type[key][r["cat"]] += 1

cats = ["correct", "type_error", "boundary_error", "type_and_boundary_error", "spurious", "missed"]
header = f"{'type':<6}" + "".join(f"{c[:12]:>14}" for c in cats)
print(header)
print("-" * len(header))
for typ in sorted(per_type):
    row = f"{typ:<6}" + "".join(f"{per_type[typ].get(c, 0):>14,}" for c in cats)
    print(row)

type         correct    type_error  boundary_err  type_and_bou      spurious        missed
------------------------------------------------------------------------------------------
LOC            8,771           261           719           274           856           538
ORG            6,735           176         1,415           227         1,362         1,355
PER            8,944           168         1,176           131           626           391


In [ ]:
confusion = Counter(
    (r["gold_type"], r["type"])
    for r in records
    if r["cat"] in ("type_error", "type_and_boundary_error")
)
print("gold -> pred : count")
for (g, p), n in confusion.most_common():
    print(f"{g:>4} -> {p:<4} : {n:,}")

gold -> pred : count
 ORG -> LOC  : 359
 LOC -> ORG  : 222
 ORG -> PER  : 194
 PER -> ORG  : 181
 PER -> LOC  : 176
 LOC -> PER  : 105


In [ ]:
len_stats = defaultdict(lambda: Counter())
for r in records:
    if r["cat"] == "correct":
        ln = r["span"][1] - r["span"][0]
        bucket = min(ln, 5)
        len_stats[bucket]["correct"] += 1
    elif r["cat"] in ("boundary_error", "type_and_boundary_error", "missed"):
        ln = r["gold_span"][1] - r["gold_span"][0]
        bucket = min(ln, 5)
        len_stats[bucket]["error"] += 1

print(f"{'span len':<10}{'correct':>10}{'error':>8}{'err rate':>10}")
for b in sorted(len_stats):
    c, e = len_stats[b]["correct"], len_stats[b]["error"]
    label = f"{b}" if b < 5 else "5+"
    print(f"{label:<10}{c:>10,}{e:>8,}{e/(c+e)*100:>9.1f}%")

span len     correct   error  err rate
1             13,696   2,684     16.4%
2              7,766   1,071     12.1%
3              1,884     884     31.9%
4                632     451     41.6%
5+               472   1,136     70.6%


In [ ]:
def count_orphan_i(tags: list[str]) -> int:
    orphans, in_entity = 0, False
    for t in tags:
        if t.startswith("B-"):
            in_entity = True
        elif t.startswith("I-"):
            if not in_entity:
                orphans += 1
        else:
            in_entity = False
    return orphans

orphans = sum(count_orphan_i(t) for t in y_true)
docs_with = sum(1 for t in y_true if count_orphan_i(t) > 0)
print(f"Orphan I-tags in hi validation labels: {orphans:,}")
print(f"Docs containing >=1 orphan: {docs_with:,} / {len(y_true):,}")
print("Note: these are structurally invalid silver labels the model is scored against.")

Orphan I-tags in hi validation labels: 0
Docs containing >=1 orphan: 0 / 13,460
Note: these are structurally invalid silver labels the model is scored against.


In [ ]:
def render(doc_id: int, span: tuple[int, int, str] | None):
    toks = tokens_raw[doc_id]
    if span is None:
        return "<none>"
    return " ".join(toks[span[0]:span[1]])

shown = 0
for r in records:
    if r["cat"] in ("boundary_error", "type_and_boundary_error", "missed") and (r.get("gold_type") == "ORG" or r["type"] == "ORG"):
        print(f"--- doc {r['doc']} | {r['cat']} ---")
        print("  gold:", render(r["doc"], r["gold_span"]))
        print("  pred:", render(r["doc"], r["span"]))
        shown += 1
        if shown >= 10:
            break

--- doc 0 | missed ---
  gold: सीडी
  pred: <none>
--- doc 8 | missed ---
  gold: राष्ट्रीय स्वच्छ विद्यालय
  pred: <none>
--- doc 12 | type_and_boundary_error ---
  gold: पंजाब सरकार
  pred: पंजाब
--- doc 12 | boundary_error ---
  gold: केंद्रीय पर्यावरण मंत्री एवं वन मंत्रालय
  pred: वन मंत्रालय
--- doc 12 | boundary_error ---
  gold: केंद्रीय पर्यावरण मंत्री एवं वन मंत्रालय
  pred: पर्यावरण
--- doc 17 | boundary_error ---
  gold: अखिल भारतीय कांग्रेस
  pred: अखिल भारतीय कांग्रेस पार्टी
--- doc 31 | missed ---
  gold: समुद्र कार्गो
  pred: <none>
--- doc 38 | boundary_error ---
  gold: भारतीय रक्षा कंपनियों - अशोक लीलैंड
  pred: अशोक लीलैंड
--- doc 42 | missed ---
  gold: बॉक्स
  pred: <none>
--- doc 44 | type_and_boundary_error ---
  gold: सुंदरबनी
  pred: सुंदरबनी कॉलेज


In [ ]:
import json

summary = {
    "eval_data": "hi validation (silver labels)",
    "model": "indicbert_hi_best",
    "category_counts": dict(cat_counts),
    "type_confusion": {f"{g}->{p}": n for (g, p), n in confusion.items()},
    "orphan_i_tags_validation": orphans,
}
path = f"{DRIVE_DIR}/error_analysis_hi.json"
with open(path, "w") as f:
    json.dump(summary, f, indent=2)
print("Saved:", path)

Saved: /content/drive/MyDrive/indic-ner/error_analysis_hi.json
